## Binary Trees · Study Notes *(DFS)*

**Author:** KTH  
**Date:** Aug 20, 2026

---
> The method of solution involves the development of a theory of finite automata
> operating on infinite trees.
>
> — "Decidability of Second Order Theories and Automata on Trees," M. O. Rabin, 1969

**Organization note.** The chapter's own ordering puts terminology before
traversals. Per request, this notebook leads with **DFS** — the three
depth-first traversals *are* the chapter's boot camp — and places the
terminology and tree-shape taxonomy after it as reference. A short **BFS**
contrast is appended at the end and clearly marked as *supplementary, not from
this chapter*.

Formally, a **binary tree** is either empty, or a root node *r* together with a
left binary tree and a right binary tree. The subtrees are themselves binary
trees. Binary trees most commonly occur in the context of **binary search
trees**, where keys are stored in sorted fashion — but at a high level, binary
trees are appropriate whenever you're **dealing with hierarchies**.

## 0. The `BinaryTreeNode` prototype

Often the node stores additional data. Its prototype:

In [1]:
class BinaryTreeNode:
    def __init__(self, data=None, left=None, right=None):
        self.data = data
        self.left = left
        self.right = right

In [2]:
# Build the tree from Figure 6.1 (letters as data) so every traversal below is runnable.
#                      A
#            B                   I
#        C       F           J       O
#      D   E       G       K           P
#            H           L   N
#                          M
D = BinaryTreeNode('D')
E = BinaryTreeNode('E')
C = BinaryTreeNode('C', D, E)
H = BinaryTreeNode('H')
G = BinaryTreeNode('G', H)
F = BinaryTreeNode('F', None, G)
B = BinaryTreeNode('B', C, F)
M = BinaryTreeNode('M')
L = BinaryTreeNode('L', None, M)
N = BinaryTreeNode('N')
K = BinaryTreeNode('K', L, N)
J = BinaryTreeNode('J', None, K)
P = BinaryTreeNode('P')
O = BinaryTreeNode('O', None, P)
I = BinaryTreeNode('I', J, O)
A = BinaryTreeNode('A', B, I)

root = A
print("root:", root.data)

root: A


# Part 1 — DFS: traversing a binary tree

A **key computation** on a binary tree is traversing all its nodes. (Traversing
is also called **walking**.) The three depth-first orders differ only in *when
the root is visited* relative to its subtrees:

| Traversal | Order | On Figure 6.1 |
|---|---|---|
| **Inorder** | left subtree → **root** → right subtree | D, C, E, B, F, H, G, A, J, L, M, K, N, I, O, P |
| **Preorder** | **root** → left subtree → right subtree | A, B, C, D, E, F, G, H, I, J, K, L, M, N, O, P |
| **Postorder** | left subtree → right subtree → **root** | D, E, C, H, G, F, B, M, L, N, K, J, P, O, I, A |

**Complexity.** Let *T* be a binary tree of *n* nodes with height *h*.
Implemented recursively, these traversals have **`O(n)` time** and **`O(h)`
additional space** — the space is dictated by the maximum depth of the function
call stack. If each node has a **parent field**, the traversals can be done with
**`O(1)` additional space**.

## 1. Binary trees boot camp

A good way to get up to speed with binary trees is to implement the three basic
traversals. Note how all three fall out of a **single recursive skeleton** — the
only difference is where the `print` sits among the two recursive calls.

In [3]:
def tree_traversal(root):
    if root:
        # Preorder: Processes the root before the traversals of left and right
        # children.
        print('Preorder: %s' % root.data)
        tree_traversal(root.left)
        # Inorder: Processes the root after the traversal of left child and
        # before the traversal of right child.
        print('Inorder: %s' % root.data)
        tree_traversal(root.right)
        # Postorder: Processes the root after the traversals of left and right
        # children.
        print('Postorder: %s' % root.data)

In [4]:
# Run it on a small subtree so the interleaved output stays readable
small = BinaryTreeNode('C', BinaryTreeNode('D'), BinaryTreeNode('E'))
tree_traversal(small)

Preorder: C
Preorder: D
Inorder: D
Postorder: D
Inorder: C
Preorder: E
Inorder: E
Postorder: E
Postorder: C


**Complexity of the boot camp code.** Time is `O(n)`, where *n* is the number
of nodes. Although no memory is explicitly allocated, the function call stack
reaches a maximum depth of *h*, the height of the tree — so **space is `O(h)`**.
The minimum value of *h* is `log n` (complete binary tree); the maximum is *n*
(skewed tree).

## 2. The three traversals, separated

Splitting the combined skeleton into three functions makes each order's output
verifiable against the table above.

In [5]:
def preorder(root, out=None):
    out = [] if out is None else out
    if root:
        out.append(root.data)          # visit root FIRST
        preorder(root.left, out)
        preorder(root.right, out)
    return out

def inorder(root, out=None):
    out = [] if out is None else out
    if root:
        inorder(root.left, out)
        out.append(root.data)          # visit root BETWEEN subtrees
        inorder(root.right, out)
    return out

def postorder(root, out=None):
    out = [] if out is None else out
    if root:
        postorder(root.left, out)
        postorder(root.right, out)
        out.append(root.data)          # visit root LAST
    return out

In [6]:
# Verify against the orders given in the chapter
print("Preorder :", ''.join(preorder(root)))
print("Inorder  :", ''.join(inorder(root)))
print("Postorder:", ''.join(postorder(root)))

assert ''.join(inorder(root))   == 'DCEBFHGAJLMKNIOP'
assert ''.join(preorder(root))  == 'ABCDEFGHIJKLMNOP'
assert ''.join(postorder(root)) == 'DECHGFBMLNKJPOIA'
print("\nAll three match the chapter's stated orders.")

Preorder : ABCDEFGHIJKLMNOP
Inorder  : DCEBFHGAJLMKNIOP
Postorder: DECHGFBMLNKJPOIA

All three match the chapter's stated orders.


## 3. Table 6.1 — Top Tips for Binary Trees

- **Recursive algorithms are well-suited to problems on trees.** Remember to
  include space *implicitly* allocated on the function call stack when doing
  space complexity analysis.
- Some tree problems have simple brute-force solutions using `O(n)` space, but
  subtler solutions that **use the existing tree nodes** to reduce space
  complexity to `O(1)`.
- Consider **left- and right-skewed trees** when doing complexity analysis. Note
  that `O(h)` complexity translates into `O(log n)` for **balanced** trees, but
  `O(n)` for **skewed** trees.
- If each node has a **parent field**, use it to make your code simpler and to
  reduce time and space complexity.
- It's easy to make the mistake of **treating a node that has a single child as
  a leaf**.

In [7]:
# The last tip, made concrete: a leaf has NO children — not "at most one".
def is_leaf(node):
    return node.left is None and node.right is None      # correct

def is_leaf_WRONG(node):
    return node.left is None or node.right is None       # treats F (one child) as a leaf

print("F has one child (G).")
print("  is_leaf(F)       =", is_leaf(F), " <- correct")
print("  is_leaf_WRONG(F) =", is_leaf_WRONG(F), " <- the classic bug")

# Chapter states the leaves of Figure 6.1 are D, E, H, M, N, P
leaves = [n.data for n in [D, E, C, F, G, H, M, N, P, A] if is_leaf(n)]
print("\nleaves found:", leaves)

F has one child (G).
  is_leaf(F)       = False  <- correct
  is_leaf_WRONG(F) = True  <- the classic bug

leaves found: ['D', 'E', 'H', 'M', 'N', 'P']


## 4. Skew and the `O(h)` warning, demonstrated

The third tip matters because *h* ranges from `log n` to `n` depending on shape.
A **left-skewed** tree has no node with a right child; a **right-skewed** tree
has no node with a left child. Either is referred to as **skewed**.

In [8]:
def height(node):
    if node is None:
        return -1                          # height of empty tree; a leaf has height 0
    return 1 + max(height(node.left), height(node.right))

# Right-skewed: every node has only a right child
skewed = BinaryTreeNode(0)
cur = skewed
for i in range(1, 8):
    cur.right = BinaryTreeNode(i)
    cur = cur.right

print("Figure 6.1 tree: height =", height(root), "(chapter says 5)")
print("  subtree at B : height =", height(B), "(chapter says 3)")
print("  subtree at H : height =", height(H), "(chapter says 0)")
print("\nSkewed tree of 8 nodes: height =", height(skewed),
      "-> recursion depth O(n), not O(log n)")

Figure 6.1 tree: height = 5 (chapter says 5)
  subtree at B : height = 3 (chapter says 3)
  subtree at H : height = 0 (chapter says 0)

Skewed tree of 8 nodes: height = 7 -> recursion depth O(n), not O(log n)


# Part 2 — Terminology and tree shapes *(reference)*

## 5. Parent, child, ancestor, descendant

Each node except the root is itself the root of a left or right subtree. If *l*
is the root of *p*'s left subtree, *l* is the **left child** of *p*, and *p* is
the **parent** of *l* (right child is similar). With the exception of the root,
**every node has a unique parent**. Usually — but not universally — the node
definition includes a **parent field** (null for the root).

For any node there exists a unique sequence of nodes from the root to that node,
each a child of the previous one: the **search path** from the root to the node.

The parent-child relationship defines an **ancestor-descendant** relationship: a
node is an **ancestor** of *d* if it lies on the search path from the root to
*d*; then *d* is a **descendant** of that node. **Convention: a node is an
ancestor and a descendant of itself.** A node with no descendants except itself
is a **leaf**.

- The **depth** of a node *n* is the number of nodes on the search path from the
  root to *n*, **not including** *n* itself.
- The **height** of a binary tree is the maximum depth of any node in it.
- A **level** of a tree is all nodes at the same depth.

**Worked examples from Figure 6.1:** Node I is the parent of J and O. Node G is
a descendant of B. The search path to L is ⟨A, I, J, K, L⟩. The depth of N is 4.
Node M has maximum depth, so the tree's height is 5. The subtree rooted at B has
height 3; the subtree rooted at H has height 0. The leaves are D, E, H, M, N,
and P.

## 6. Tree shapes

| Shape | Definition |
|---|---|
| **Full** | Every node other than the leaves has **two** children. |
| **Perfect** | A full binary tree in which all leaves are at the **same depth**, and every parent has two children. |
| **Complete** | Every level except possibly the last is completely filled, and all nodes are **as far left as possible**. |
| **Skewed** | **Left-skewed:** no node has a right child. **Right-skewed:** no node has a left child. |

⚠️ *This terminology is not universal* — some authors use "complete binary tree"
where this book writes "perfect binary tree."

**Countable facts** (the first is straightforward to prove by induction):
- The number of **nonleaf** nodes in a **full** binary tree is **one less than
  the number of leaves**.
- A **perfect** binary tree of height *h* contains exactly **2^(h+1) − 1** nodes,
  of which **2^h** are leaves.
- A **complete** binary tree on *n* nodes has height **⌊log n⌋**.

In [9]:
import math

# Verify the perfect-tree formulas
for h in range(5):
    nodes, leaves = 2**(h + 1) - 1, 2**h
    print(f"perfect tree, height {h}: {nodes:>3} nodes, {leaves:>2} leaves, "
          f"{nodes - leaves:>2} nonleaf  (nonleaf == leaves - 1: {nodes - leaves == leaves - 1})")

print()
for n in (1, 2, 4, 7, 8, 16):
    print(f"complete tree on {n:>2} nodes -> height {math.floor(math.log2(n))}")

perfect tree, height 0:   1 nodes,  1 leaves,  0 nonleaf  (nonleaf == leaves - 1: True)
perfect tree, height 1:   3 nodes,  2 leaves,  1 nonleaf  (nonleaf == leaves - 1: True)
perfect tree, height 2:   7 nodes,  4 leaves,  3 nonleaf  (nonleaf == leaves - 1: True)
perfect tree, height 3:  15 nodes,  8 leaves,  7 nonleaf  (nonleaf == leaves - 1: True)
perfect tree, height 4:  31 nodes, 16 leaves, 15 nonleaf  (nonleaf == leaves - 1: True)

complete tree on  1 nodes -> height 0
complete tree on  2 nodes -> height 1
complete tree on  4 nodes -> height 2
complete tree on  7 nodes -> height 2
complete tree on  8 nodes -> height 3
complete tree on 16 nodes -> height 4


# Part 3 — BFS 

Chapter 6's boot camp covers only the depth-first traversals. **Breadth-first**
(level-order) traversal appears in this book as **problem 5.2** in the Stacks and
Queues chapter — "Compute binary tree nodes in order of increasing depth" —
because it's an application of a *queue* rather than of recursion. Included here
for contrast.

**The distinction that matters:**

| | DFS | BFS |
|---|---|---|
| Data structure | Call stack (or explicit stack) | **Queue** |
| Auxiliary space | `O(h)` — tree height | `O(w)` — tree **width** (max nodes on a level) |
| Natural for | Path/subtree questions, structural recursion | Shortest path in edges, level-by-level output |

Note that the space complexities are **not** comparable in general: for a
balanced tree the last level holds ~n/2 nodes, so BFS is `O(n)` while DFS is
`O(log n)`; for a skewed tree BFS is `O(1)` while DFS is `O(n)`.

In [10]:
import collections

def bfs_levels(root):
    '''Return a list of levels, each a list of node data at that depth.'''
    if not root:
        return []
    result = []
    q = collections.deque([root])
    while q:
        level = []
        for _ in range(len(q)):        # snapshot the level's size before expanding
            node = q.popleft()
            level.append(node.data)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        result.append(level)
    return result

for depth, level in enumerate(bfs_levels(root)):
    print(f"depth {depth}: {' '.join(level)}")

depth 0: A
depth 1: B I
depth 2: C F J O
depth 3: D E G K P
depth 4: H L N
depth 5: M
